# Foundational Agentic Workflows

### Live Demo
You can experience the live version of my **Personal Portfolio Assistant** on Hugging Face Spaces:  
-> **[Career Conversation App](https://huggingface.co/spaces/mikeaig4real/career_conversation)**

### In This Notebook
1. **Task Management Tools**: Creating and marking steps as done.
2. **Agentic Run-Loop**: An iterative process where the LLM can use tools to plan and execute tasks.
3. **Algorithm Solver**: A demonstration of the agent solving a **Binary Search** implementation step-by-step using pseudo-code.

> [!TIP]
> Run all cells in sequence to see the agent iteratively build and complete its plan!

In [1]:
import os
from rich.console import Console
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

In [ ]:
BASE_URL="https://openrouter.ai/api/v1"
API_KEY=os.getenv("OPENROUTER_API_KEY")
API_KEY_PREFIX="sk-or-v1-"
if not API_KEY or not API_KEY.startswith(API_KEY_PREFIX):
    print("OPENROUTER_API_KEY not properly configured")
else:
    print("OPENROUTER_API_KEY properly configured")

In [ ]:
ai=OpenAI(base_url=BASE_URL, api_key=API_KEY)

In [ ]:
steps = []
completed = []

In [ ]:
samples=["Loop through the array", "check if an item is the one", "if it is return the index"]
steps.extend(samples)
completed.extend([False] * len(samples))

steps, completed

In [ ]:
def beautify_log(msg=""):
    try:
        Console().print(msg)
    except Exception:
        print(msg)



def get_steps_update() -> str:
    result=""
    for i, done in enumerate(completed):
        if completed[i]:
            result+=f"Step #{i + 1}: [green][strike]{steps[i]}[/strike][/green]\n"
        else:
            result+=f"Step #{i + 1}: {steps[i]}\n"
    beautify_log(result)
    return result

def create_steps(descriptions: list[str]) -> str:
    steps.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_steps_update()

def mark_step_as_done(index: int, side_note: str = "") -> str:
    if 0 <= index < len(completed):
        completed[index] = True
        if side_note:
            beautify_log(side_note)
    return get_steps_update()


In [ ]:
create_steps_tool = {
    "type": "function",
    "function": {
        "name": "create_steps",
        "description": "Create one or more task steps and return the updated steps visualization.",
        "parameters": {
            "type": "object",
            "properties": {
                "descriptions": {
                    "type": "array",
                    "items": { "type": "string" },
                    "description": "The descriptions of the steps to create."
                }
            },
            "required": ["descriptions"]
        }
    }
}

mark_step_as_done_tool = {
    "type": "function",
    "function": {
        "name": "mark_step_as_done",
        "description": "Mark a specific task step as completed and return the updated steps visualization.",
        "parameters": {
            "type": "object",
            "properties": {
                "index": {
                    "type": "integer",
                    "description": "The 0-based index of the step to mark as done."
                },
                "side_note": {
                    "type": "string",
                    "description": "Optional note regarding the completion of the step."
                }
            },
            "required": ["index"]
        }
    }
}

tools = [create_steps_tool, mark_step_as_done_tool]


In [ ]:
import json

def handle_tools(tool_calls: list) -> list[dict]:
    tool_map = {
        "create_steps": create_steps,
        "mark_step_as_done": mark_step_as_done
    }
    results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        function_to_call = tool_map.get(function_name)
        if function_to_call:
            result = function_to_call(**arguments)
        else:
            result = f"Error: Function {function_name} not found."
            
        results.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": result
        })
    return results


In [ ]:
MODEL = "openai/gpt-4o"

def run_loop(messages: list[dict]) -> str:
    is_running = True
    
    while is_running:
        response = ai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            reasoning_effort=None
        )
        
        choice = response.choices[0]
        message = choice.message
        messages.append(message)
        
        if choice.finish_reason == "tool_calls":
            tool_results = handle_tools(message.tool_calls)
            messages.extend(tool_results)
        else:
            is_running = False
            
    final_content = messages[-1].content
    if final_content:
        beautify_log(final_content)
    return final_content


In [ ]:
# Reset state
steps = []
completed = []

system_prompt = """You are an algorithm expert. Your goal is to solve algorithms in a step-by-step progressive manner using the provided tools. 
For each step, you should provide pseudo-code and update or refactor it as needed. 
Use the `create_steps` tool to define your plan and `mark_step_as_done` as you complete each task. 
Always respond with pseudo-code, never actual executable code."""

user_prompt = "Please solve the Binary Search algorithm step-by-step using pseudo-code and the provided tools."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Run the loop
final_result = run_loop(messages)
